## Creación de Base clean

In [15]:
import pandas as pd
import numpy as np
import optuna
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import holidays

# Configura Pandas para mostrar floats sin notación científica
pd.set_option('display.float_format', lambda x: '%.0f' % x)

In [16]:
path_orders  = "bases/Orders.csv"
path_details = "bases/details_clean.csv"
path_results = "bases/resultados_sin_hash.csv"

df_orders    = pd.read_csv(path_orders, dtype={'id_pedido': str})
df_details   = pd.read_csv(path_details, dtype={'id_pedido': str})
df_resultados = pd.read_csv(path_results, dtype={'id_pedido': str})


In [17]:
# Forzar a los tres DataFrames a ser flotantes numéricos
df_orders['id_pedido'] = df_orders['id_pedido'].astype(float)
df_details['id_pedido'] = df_details['id_pedido'].astype(float)
df_resultados['id_pedido'] = df_resultados['id_pedido'].astype(float)

In [18]:
df_orders['customer_id'] = df_orders['customer_id'].astype(float)

In [19]:
df_orders['fecha_pedido'] = pd.to_datetime(df_orders['fecha_pedido'], errors='coerce')

/var/folders/gc/wcr1rzm57ls2jc6mkw7j8xr80000gn/T/ipykernel_34694/1516480952.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_orders['fecha_pedido'] = pd.to_datetime(df_orders['fecha_pedido'], errors='coerce')


In [20]:
df_orders = df_orders[df_orders['pais'] == 'México']

In [21]:
# Mantener unicamente el primer registro de cada id_pedido en df_orders
df_orders = df_orders.drop_duplicates(subset='id_pedido', keep='first')

In [22]:
# Dropear fecha_entrega
df_orders = df_orders.drop(columns=['fecha_entrega'], errors='ignore')

In [23]:
# 1. Definimos el rango completo de días posibles entre 2022 y 2025
rango_fechas = pd.date_range(start='2022-01-01', end='2025-12-31', freq='h')

# 2. Creamos la columna 'fecha_pedido' eligiendo fechas al azar de ese rango
#    usamos len(df_orders) para generar una fecha para cada fila de tu DataFrame
df_orders['fecha_pedido'] = np.random.choice(rango_fechas, size=len(df_orders))

In [24]:
df_orders.fecha_pedido.value_counts()

fecha_pedido
2023-08-26 04:00:00    6
2022-10-12 20:00:00    6
2022-12-10 22:00:00    6
2022-07-07 05:00:00    6
2025-03-20 15:00:00    6
                      ..
2023-12-31 20:00:00    1
2023-03-14 20:00:00    1
2022-05-26 23:00:00    1
2024-04-04 04:00:00    1
2025-04-24 18:00:00    1
Name: count, Length: 18961, dtype: int64

In [25]:
for df in [df_orders, df_details, df_resultados]:
    df['id_pedido'] = pd.to_numeric(df['id_pedido'], errors='coerce')

In [26]:
# Merge para la base completa usada en demo/modelo XGBoost (~957k registros):
# 1) Orders + Details por id_pedido: baja informacion del pedido a cada linea.
# 2) Resultados por id_pedido: conserva el comportamiento historico de pedidos con sustitucion.
#
# Nota tecnica: esta base puede repetir id_linea porque un mismo pedido puede tener
# mas de un resultado de sustitucion. Para el entregable de Streamlit usamos esta
# base completa porque refleja el universo de ~957k registros que se desea mostrar.

cols_cambio = [
    'id_pedido',
    'sku_solicitado_cambio',
    'nombre_sku_solicitado_cambio',
]

df_resultados_cambios = df_resultados[cols_cambio].drop_duplicates().copy()

df = (
    df_orders
    .merge(df_details, on='id_pedido', how='left')
    .merge(df_resultados_cambios, on='id_pedido', how='left')
)

# Nos quedamos con lineas validas.
df = df.dropna(subset=['id_linea']).copy()

# Normalizar nombres para que app/modelo esperen columnas consistentes.
df = df.rename(columns={
    'Quantity': 'quantity',
    'Status': 'status',
})

# Target: pedido/linea asociada a un pedido donde hubo sustitucion historica.
df['fue_sustituida'] = df['sku_solicitado_cambio'].notna().astype(int)

# Features simples de tiempo para el modelo.
df['fecha_pedido'] = pd.to_datetime(df['fecha_pedido'], errors='coerce')
df['hora'] = df['fecha_pedido'].dt.hour
df['dia_semana'] = df['fecha_pedido'].dt.dayofweek
df['mes'] = df['fecha_pedido'].dt.month
df['es_fin_semana'] = (df['dia_semana'] >= 5).astype(int)

# Guardamos version completa para auditoria y para app/modelo.
df.to_csv('bases/merged.csv', index=False)
df.to_csv('bases/merged_final.csv', index=False)

print('merged completo:', df.shape)
print('id_pedido unicos:', df['id_pedido'].nunique())
print('id_linea unicos:', df['id_linea'].nunique())
print('id_linea duplicados:', df['id_linea'].duplicated().sum())
print('filas completas duplicadas:', df.duplicated().sum())
print('columnas _x/_y:', [col for col in df.columns if col.endswith(('_x', '_y'))])
print(df['fue_sustituida'].value_counts())


merged completo: (957673, 23)
id_pedido unicos: 27349
id_linea unicos: 935557
id_linea duplicados: 22116
filas completas duplicadas: 0
columnas _x/_y: []
fue_sustituida
0    773637
1    184036
Name: count, dtype: int64
